**Extracting absolute counts for Representative Days**

In [11]:
import pandas as pd
from pathlib import Path

# ============================================================
# SETTINGS
# ============================================================

BASE_DIR = Path(
    r"C:\Users\mogul\OneDrive\Masaüstü\mt_emre\detector_data_aktuell"
)

IN_FILE = BASE_DIR / "detector_data_CSV_new" / "all_intersections_common_days_long.csv"

OUT_DIR = BASE_DIR / "representative_day_routesampler_base"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SCENARIOS = [
    {
        "name": "weekday_morning",
        "date": "2026-03-10",
        "start_time": "08:00:00",
        "end_time": "09:00:00"
    },
    {
        "name": "weekday_evening",
        "date": "2026-03-23",
        "start_time": "16:00:00",
        "end_time": "17:00:00"
    },
    {
        "name": "weekend_morning",
        "date": "2026-03-14",
        "start_time": "08:00:00",
        "end_time": "09:00:00"
    },
    {
        "name": "weekend_evening",
        "date": "2026-03-22",
        "start_time": "16:00:00",
        "end_time": "17:00:00"
    }
]

# ============================================================
# LOAD
# ============================================================

df = pd.read_csv(IN_FILE)

# robust timestamp handling
if "timestamp_local" in df.columns:
    df["timestamp_local"] = pd.to_datetime(df["timestamp_local"], errors="coerce")
elif "timestamp" in df.columns:
    df["timestamp_local"] = pd.to_datetime(df["timestamp"], errors="coerce")
else:
    raise ValueError("No timestamp column found. Expected 'timestamp_local' or 'timestamp'.")

if "timestamp_utc" in df.columns:
    df["timestamp_utc"] = pd.to_datetime(df["timestamp_utc"], errors="coerce", utc=True)
else:
    df["timestamp_utc"] = pd.NaT

df["date"] = pd.to_datetime(df["date"], errors="coerce").dt.strftime("%Y-%m-%d")

if "time" not in df.columns:
    df["time"] = df["timestamp_local"].dt.strftime("%H:%M:%S")

df["count"] = pd.to_numeric(df.get("count", 0), errors="coerce").fillna(0)
df["occupancy"] = pd.to_numeric(df.get("occupancy", 0), errors="coerce").fillna(0)

# new files use speed, but keep compatibility with old velocity naming
if "speed" in df.columns:
    df["speed"] = pd.to_numeric(df["speed"], errors="coerce").fillna(0)
elif "velocity" in df.columns:
    df["speed"] = pd.to_numeric(df["velocity"], errors="coerce").fillna(0)
else:
    df["speed"] = 0.0

# create begin_s / end_s from time if not present
if "begin_s" not in df.columns:
    df["begin_s"] = (
        df["timestamp_local"].dt.hour * 3600 +
        df["timestamp_local"].dt.minute * 60 +
        df["timestamp_local"].dt.second
    )

if "end_s" not in df.columns:
    # your data is 15-minute intervals
    df["end_s"] = df["begin_s"] + 900

df["begin_s"] = pd.to_numeric(df["begin_s"], errors="coerce")
df["end_s"] = pd.to_numeric(df["end_s"], errors="coerce")

# add missing columns for compatibility with old workflow
for col in ["edge_id", "movement", "count_type", "from_edge", "to_edge"]:
    if col not in df.columns:
        df[col] = pd.NA

# ============================================================
# EXTRACT PER SCENARIO
# ============================================================

summary_rows = []

for sc in SCENARIOS:
    name = sc["name"]
    rep_day = sc["date"]
    start_time = sc["start_time"]
    end_time = sc["end_time"]

    temp = df[
        (df["date"] == rep_day) &
        (df["time"] >= start_time) &
        (df["time"] < end_time)
    ].copy()

    temp = temp.sort_values(
        ["intersection", "timestamp_local", "detector_id"]
    ).reset_index(drop=True)

    # 1) raw long representative-day base
    keep_cols = [
        "intersection",
        "date",
        "timestamp_utc",
        "timestamp_local",
        "time",
        "begin_s",
        "end_s",
        "detector_id",
        "sumo_id",
        "approach",
        "edge_id",
        "movement",
        "count_type",
        "from_edge",
        "to_edge",
        "count",
        "occupancy",
        "speed",
    ]

    keep_cols_existing = [c for c in keep_cols if c in temp.columns]

    long_out = OUT_DIR / f"{name}_routesampler_base_long.csv"
    temp[keep_cols_existing].to_csv(long_out, index=False, encoding="utf-8-sig")

    # 2) detector intervals aggregated
    group_cols = [
        "intersection", "date", "begin_s", "end_s",
        "detector_id", "sumo_id", "approach",
        "edge_id", "count_type", "from_edge", "to_edge", "movement"
    ]
    group_cols_existing = [c for c in group_cols if c in temp.columns]

    detector_interval = (
        temp.groupby(group_cols_existing, dropna=False, as_index=False)
        .agg(
            count=("count", "sum"),
            mean_occupancy=("occupancy", "mean"),
            mean_speed=("speed", "mean")
        )
        .sort_values(["begin_s", "intersection", "detector_id"])
    )

    detector_interval_out = OUT_DIR / f"{name}_detector_interval_counts.csv"
    detector_interval.to_csv(detector_interval_out, index=False, encoding="utf-8-sig")

    # 3) 15-min detector matrix
    detector_15 = (
        temp.groupby(group_cols_existing, dropna=False, as_index=False)
        .agg(
            count=("count", "sum"),
            mean_occupancy=("occupancy", "mean"),
            mean_speed=("speed", "mean")
        )
        .sort_values(["intersection", "detector_id", "begin_s"])
    )

    detector_15_out = OUT_DIR / f"{name}_detector_15min_counts.csv"
    detector_15.to_csv(detector_15_out, index=False, encoding="utf-8-sig")

    # 4) wide matrix for quick inspection
    pivot_index = [c for c in ["intersection", "detector_id", "sumo_id", "approach", "edge_id", "count_type", "from_edge", "to_edge", "movement"] if c in temp.columns]

    pivot = temp.pivot_table(
        index=pivot_index,
        columns="time",
        values="count",
        aggfunc="sum",
        fill_value=0
    ).reset_index()

    pivot_out = OUT_DIR / f"{name}_detector_time_matrix.csv"
    pivot.to_csv(pivot_out, index=False, encoding="utf-8-sig")

    # 5) technical summary by detector
    summary_group_cols = [c for c in ["intersection", "detector_id", "sumo_id", "approach", "edge_id", "count_type", "from_edge", "to_edge", "movement"] if c in temp.columns]

    detector_summary = (
        temp.groupby(summary_group_cols, dropna=False, as_index=False)
        .agg(
            total_count=("count", "sum"),
            mean_occupancy=("occupancy", "mean"),
            mean_speed=("speed", "mean"),
            n_intervals=("count", "size")
        )
        .sort_values(["intersection", "detector_id"])
    )

    detector_summary_out = OUT_DIR / f"{name}_detector_summary.csv"
    detector_summary.to_csv(detector_summary_out, index=False, encoding="utf-8-sig")

    summary_rows.append({
        "scenario": name,
        "date": rep_day,
        "start_time": start_time,
        "end_time": end_time,
        "n_rows": len(temp),
        "n_detectors": temp[["intersection", "detector_id"]].drop_duplicates().shape[0] if not temp.empty else 0,
        "total_count": temp["count"].sum(),
        "mean_occupancy_overall": temp["occupancy"].mean() if not temp.empty else 0
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(
    OUT_DIR / "representative_day_routesampler_base_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\n============================================================")
print("REPRESENTATIVE DAY EXTRACTION SUMMARY")
print("============================================================")
print(summary_df.to_string(index=False))
print(f"\nSaved outputs to:\n{OUT_DIR}")


REPRESENTATIVE DAY EXTRACTION SUMMARY
       scenario       date start_time end_time  n_rows  n_detectors  total_count  mean_occupancy_overall
weekday_morning 2026-03-10   08:00:00 09:00:00     196           49         9295               34.443878
weekday_evening 2026-03-23   16:00:00 17:00:00     196           49        10763               42.387755
weekend_morning 2026-03-14   08:00:00 09:00:00     196           49         7017               23.693878
weekend_evening 2026-03-22   16:00:00 17:00:00     196           49         8046               26.479592

Saved outputs to:
C:\Users\mogul\OneDrive\Masaüstü\mt_emre\detector_data_aktuell\representative_day_routesampler_base


After identifying the representative days, the original absolute detector counts were extracted for each selected scenario. This step was necessary because the K-means clustering was based on normalized daily profiles only for pattern recognition, whereas the subsequent demand reconstruction requires absolute traffic volumes.

For each representative day, the detector data were filtered to the corresponding scenario period and aggregated by detector. In addition, the full 15-minute detector time series of the selected day was preserved for later calibration and validation. These aggregated absolute counts form the basis for the subsequent mapping of detector observations to SUMO network edges and for the generation of demand inputs for route-based traffic reconstruction.

***detector → edge → counts.xml***

In [14]:
import pandas as pd
import xml.etree.ElementTree as ET
from pathlib import Path
from xml.dom import minidom

# ============================================================
# SETTINGS
# ============================================================

BASE_DIR = Path(r"C:\Users\mogul\OneDrive\Masaüstü\mt_emre\detector_data_aktuell")

LONG_FILE = BASE_DIR / "detector_data_CSV_new" / "all_intersections_common_days_long.csv"
REP_DAY_FILE = BASE_DIR / "detector_data_CSV_new" / "kmeans_representative_days" / "final_representative_day_summary.csv"

OUT_DIR = BASE_DIR / "counts_occupancy_on_edges"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SCENARIOS = [
    {"name": "weekday_morning", "begin": 0.0, "end": 3600.0, "start_time": "08:00:00", "end_time": "09:00:00"},
    {"name": "weekday_evening", "begin": 0.0, "end": 3600.0, "start_time": "16:00:00", "end_time": "17:00:00"},
    {"name": "weekend_morning", "begin": 0.0, "end": 3600.0, "start_time": "08:00:00", "end_time": "09:00:00"},
    {"name": "weekend_evening", "begin": 0.0, "end": 3600.0, "start_time": "16:00:00", "end_time": "17:00:00"},
]

# ============================================================
# EDGE COUNT / OCCUPANCY PART
# ============================================================

EDGE_RULES = [
    # ---------------- LSA16 ----------------
    {"name": "LSA16_to_LSA10",     "members": [("LD-LSA16", 13)],                    "edge": "-202157700#5"},
    {"name": "LSA16_W_exit",       "members": [("LD-LSA16", 15)],                    "edge": "26202222#0"},
    {"name": "LSA16_S_approach",   "members": [("LD-LSA16", 1), ("LD-LSA16", 2), ("LD-LSA16", 33)], "edge": "-202157703"},
    {"name": "LSA16_N_approach",   "members": [("LD-LSA16", 6), ("LD-LSA16", 7)],   "edge": "-1182936853"},
    {"name": "LSA16_W_stopline",   "members": [("LD-LSA16", 3), ("LD-LSA16", 4)],   "edge": "-26202222#0.41"},
    {"name": "LSA16_E_approach",   "members": [("LD-LSA16", 10), ("LD-LSA16", 11)], "edge": "202157700#3"},
  

    # ---------------- LSA10 ----------------
    {"name": "LSA10_S_approach",   "members": [("LD-LSA10", 31)],                    "edge": "-1183449794"},
    {"name": "LSA10_E_approach",   "members": [("LD-LSA10", 3), ("LD-LSA10", 4)],   "edge": "75642281"},
    {"name": "LSA10_E_approach",   "members": [("LD-LSA10", 26), ("LD-LSA10", 27)],   "edge": "-13831317#2.34"},
    #{"name": "LSA10_NW_approach",  "members": [("LD-LSA10", 2)],                     "edge": "-13831317#2"},
    {"name": "LSA10_NE_approach",  "members": [("LD-LSA10", 29)],                    "edge": "-25563346#2"},

    # ---------------- LSA1 ----------------
    {"name": "LSA1_W_approach",    "members": [("LD-LSA1", 7), ("LD-LSA1", 9)],      "edge": "-202234344"},
    {"name": "LSA1_E_approach",    "members": [("LD-LSA1", 24), ("LD-LSA1", 26), ("LD-LSA1", 14)], "edge": "-237921040"},
    {"name": "LSA1_N_approach",    "members": [("LD-LSA1", 40), ("LD-LSA1", 28), ("LD-LSA1", 30)], "edge": "237921036.41"},

    # ---------------- LSA9 ----------------
    {"name": "LSA9_W_approach",    "members": [("LD-LSA9", 1), ("LD-LSA9", 2)],      "edge": "-30329931#2"},
    {"name": "LSA9_E_approach",    "members": [("LD-LSA9", 8), ("LD-LSA9", 9)],      "edge": "202234068#0"},
    {"name": "LSA9_NE_approach",   "members": [("LD-LSA9", 6), ("LD-LSA9", 7)],      "edge": "-14811383#2"},
    {"name": "LSA9_S_approach",   "members": [("LD-LSA9", 10)],   "edge": "-14811373#0"},
]

# ============================================================
# TURN COUNT / OCCUPANCY PART
# ============================================================

TURN_RULES = [
    # ---------------- LSA16 ----------------
    {"detector": ("LD-LSA16", 4),  "from": "-26202222#0.41", "to": ["893429999#1"]},
    {"detector": ("LD-LSA16", 3),  "from": "-26202222#0.41", "to": ["-202157700#5"]},
    {"detector": ("LD-LSA16", 1),  "from": "-202157703",     "to": ["-202157700#5"]},
    {"detector": ("LD-LSA16", 2),  "from": "-202157703",     "to": ["893429999#1" , "-202157700#5"]},
    {"detector": ("LD-LSA16", 33), "from": "-202157703",     "to": ["893429999#1"]},
    {"detector": ("LD-LSA16", 6),  "from": "-1182936853",    "to": ["26202222#0"]},
    {"detector": ("LD-LSA16", 7),  "from": "-1182936853",    "to": ["12528053#0" , "26202222#0" ]},
    {"detector": ("LD-LSA16", 11), "from": "202157700#3",    "to": ["12528053#0"]},
    {"detector": ("LD-LSA16", 10), "from": "202157700#3",    "to": ["893429999#1", "26202222#0"]},

    # ---------------- LSA10 ----------------
    {"detector": ("LD-LSA10", 27), "from": "-13831317#2.34", "to": ["25563346#0", "-84620353#2"]},
    {"detector": ("LD-LSA10", 26), "from": "-13831317#2.34", "to": ["17707991#0", "14811377"]},
    {"detector": ("LD-LSA10", 29), "from": "-25563346#2",    "to": ["13831317#0"]},

    # ---------------- LSA1 ----------------
    #{"detector": ("LD-LSA1", 7),   "from": "-202234344",     "to": ["-237921036"]},
    {"detector": ("LD-LSA1", 9),   "from": "-202234344",     "to": ["59617619", "-281402235#1"]},
    {"detector": ("LD-LSA1", 27),  "from": "-59617619",      "to": ["-237921036"]},
    {"detector": ("LD-LSA1", 13),  "from": "-59617619",      "to": ["12695647"]},
    {"detector": ("LD-LSA1", 23),  "from": "-59617619",      "to": ["12695647"]},
    {"detector": ("LD-LSA1", 26),  "from": "-237921040",     "to": ["-281402232"], "via": {"-281402232": "-59617619 -281402235#1"}},
    {"detector": ("LD-LSA1", 40),  "from": "237921036.41",   "to": ["12695647"]},
    {"detector": ("LD-LSA1", 28),  "from": "237921036.41",   "to": ["-281402235#1"]},
    {"detector": ("LD-LSA1", 30),  "from": "237921036.41",   "to": ["59617619"]},

    # ---------------- LSA9 ----------------
   # ---------------- LSA9 ----------------
{"detector": ("LD-LSA9", 2), "from": "-30329931#2", "to": ["202234067", "-202234068#2"], "via": {"202234067": "-30329931#1 14811383#1","-202234068#2": "-30329931#1" }},
{"detector": ("LD-LSA9", 1), "from": "-30329931#2", "to": ["14811373#0", "-202234068#2"], "via": { "14811373#0": "-30329931#1", "-202234068#2": "-30329931#1"}},
{"detector": ("LD-LSA9", 9),   "from": "202234068#0",    "to": ["30329931#0"]},
{"detector": ("LD-LSA9", 8),   "from": "202234068#0",    "to": ["308681382#0", "202234067", "30329931#0"], "via": {"202234067": "14811383#1"}},
{"detector": ("LD-LSA9", 4),   "from": "-14811383#2",    "to": ["308681382#0", "30329931#0"]},
{"detector": ("LD-LSA9", 5),   "from": "-14811383#2",    "to": ["14811373#0", "-202234068#2"]},
]

# ============================================================
# STARTING SPLIT RATIOS
# ============================================================

TURN_SPLIT_RATIOS = {
    ("LD-LSA16", 10): [0.2, 0.8],
    ("LD-LSA16", 7): [0.7, 0.3],
    ("LD-LSA16", 2): [0.8, 0.2],
    ("LD-LSA10", 27): [0.4, 0.6],
    ("LD-LSA10", 26): [0.7, 0.3],
    ("LD-LSA1", 9):   [0.8, 0.2],
    ("LD-LSA9", 2):   [0.5, 0.5],
    ("LD-LSA9", 1):   [0.5, 0.5],
    ("LD-LSA9", 8):   [0.33, 0.34, 0.33],
    ("LD-LSA9", 4):   [0.5, 0.5],
    ("LD-LSA9", 5):   [0.5, 0.5],
}

# ============================================================
# HELPERS
# ============================================================

def prettify(elem):
    rough = ET.tostring(elem, encoding="utf-8")
    reparsed = minidom.parseString(rough)
    return reparsed.toprettyxml(indent="  ")

def split_value(total_value, n_parts, ratios=None):
    total_value = float(total_value)
    if ratios is None:
        return [total_value / n_parts] * n_parts
    if len(ratios) != n_parts:
        raise ValueError(f"Ratio count mismatch: {len(ratios)} vs {n_parts}")
    ratio_sum = sum(ratios)
    ratios = [r / ratio_sum for r in ratios]
    return [total_value * r for r in ratios]

def split_count(total_count, n_parts, ratios=None):
    total_count = float(total_count)
    if ratios is None:
        ratios = [1.0 / n_parts] * n_parts
    if len(ratios) != n_parts:
        raise ValueError(f"Ratio count mismatch: {len(ratios)} vs {n_parts}")

    ratio_sum = sum(ratios)
    ratios = [r / ratio_sum for r in ratios]

    raw = [total_count * r for r in ratios]
    ints = [int(x) for x in raw]
    remainder = int(round(total_count - sum(ints)))

    frac_order = sorted(range(n_parts), key=lambda i: raw[i] - ints[i], reverse=True)
    for i in range(remainder):
        ints[frac_order[i % n_parts]] += 1
    return ints

def build_detector_summary_for_day(df_long, rep_day, start_time, end_time):
    temp = df_long.copy()
    temp["date"] = pd.to_datetime(temp["date"], errors="coerce").dt.strftime("%Y-%m-%d")

    if "timestamp_local" in temp.columns:
        temp["timestamp_used"] = pd.to_datetime(temp["timestamp_local"], errors="coerce")
    else:
        temp["timestamp_used"] = pd.to_datetime(temp["timestamp"], errors="coerce")

    temp["time"] = temp["timestamp_used"].dt.strftime("%H:%M:%S")

    temp = temp[temp["date"] == rep_day].copy()
    temp = temp[(temp["time"] >= start_time) & (temp["time"] < end_time)].copy()

    temp["count"] = pd.to_numeric(temp["count"], errors="coerce").fillna(0)
    temp["occupancy"] = pd.to_numeric(temp["occupancy"], errors="coerce").fillna(0)

    summary = (
        temp.groupby(["intersection", "detector_id"], as_index=False)
        .agg(
            total_count=("count", "sum"),
            mean_occupancy=("occupancy", "mean"),
            n_intervals=("count", "size")
        )
    )

    return summary

def get_detector_total(df, detector_key, value_col="total_count"):
    temp = df[(df["intersection"] == detector_key[0]) & (df["detector_id"] == detector_key[1])]
    if temp.empty or value_col not in temp.columns:
        return 0.0
    return float(pd.to_numeric(temp[value_col], errors="coerce").fillna(0).sum())

def get_detector_mean(df, detector_key, value_col="mean_occupancy"):
    temp = df[(df["intersection"] == detector_key[0]) & (df["detector_id"] == detector_key[1])]
    if temp.empty or value_col not in temp.columns:
        return 0.0
    vals = pd.to_numeric(temp[value_col], errors="coerce").dropna()
    if vals.empty:
        return 0.0
    return float(vals.mean())

# ============================================================
# LOAD INPUTS
# ============================================================

df_long = pd.read_csv(LONG_FILE)
rep_df = pd.read_csv(REP_DAY_FILE)

if "final_representative_day" not in rep_df.columns:
    raise ValueError("Column 'final_representative_day' not found in representative day summary.")

rep_day_map = dict(zip(rep_df["scenario"], rep_df["final_representative_day"]))

# ============================================================
# MAIN
# ============================================================

for sc in SCENARIOS:
    print("\n" + "=" * 70)
    print(f"SCENARIO: {sc['name']}")
    print("=" * 70)

    if sc["name"] not in rep_day_map:
        print(f"Representative day missing for scenario: {sc['name']}")
        continue

    rep_day = str(rep_day_map[sc["name"]])
    print(f"Representative day: {rep_day}")

    df = build_detector_summary_for_day(
        df_long=df_long,
        rep_day=rep_day,
        start_time=sc["start_time"],
        end_time=sc["end_time"]
    )

    if df.empty:
        print("No data found for this scenario/day/time window.")
        continue

    scenario_summary_path = OUT_DIR / f"{sc['name']}_detector_summary.csv"
    df.to_csv(scenario_summary_path, index=False, encoding="utf-8-sig")
    print("Saved:", scenario_summary_path)

    # ---------------- EDGE DATA ----------------
    edge_rows = []
    edge_debug_rows = []

    for rule in EDGE_RULES:
        total_count = 0.0
        occ_values = []

        for det_key in rule["members"]:
            det_count = get_detector_total(df, det_key, "total_count")
            det_occ = get_detector_mean(df, det_key, "mean_occupancy")

            total_count += det_count
            occ_values.append(det_occ)

            edge_debug_rows.append({
                "rule_name": rule["name"],
                "edge": rule["edge"],
                "intersection": det_key[0],
                "detector_id": det_key[1],
                "detector_total_count": det_count,
                "detector_mean_occupancy": det_occ
            })

        mean_occ = float(sum(occ_values) / len(occ_values)) if occ_values else 0.0

        edge_rows.append({
            "edge": rule["edge"],
            "count": int(round(total_count)),
            "occupancy_mean": mean_occ,
            "rule_name": rule["name"]
        })

    edge_df_raw = pd.DataFrame(edge_rows)

    edge_counts = (
        edge_df_raw.groupby("edge", as_index=False)
        .agg(
            count=("count", "sum"),
            occupancy_mean=("occupancy_mean", "mean")
        )
        .sort_values("edge")
    )

    edge_debug_df = pd.DataFrame(edge_debug_rows)

    print("\nEDGE COUNTS + OCCUPANCY")
    print(edge_counts.to_string(index=False))

    edge_root = ET.Element("data")
    edge_interval = ET.SubElement(edge_root, "interval", {
        "id": sc["name"],
        "begin": str(sc["begin"]),
        "end": str(sc["end"])
    })

    for _, r in edge_counts.iterrows():
        ET.SubElement(edge_interval, "edge", {
            "id": str(r["edge"]),
            "entered": str(int(r["count"]))
        })

    edge_xml_path = OUT_DIR / f"edgeData_{sc['name']}.xml"
    edge_csv_path = OUT_DIR / f"edgeData_{sc['name']}.csv"
    edge_debug_path = OUT_DIR / f"edgeData_debug_{sc['name']}.csv"

    edge_counts.to_csv(edge_csv_path, index=False, encoding="utf-8-sig")
    edge_debug_df.to_csv(edge_debug_path, index=False, encoding="utf-8-sig")

    with open(edge_xml_path, "w", encoding="utf-8") as f:
        f.write(prettify(edge_root))

    print("Saved:", edge_csv_path)
    print("Saved:", edge_debug_path)
    print("Saved:", edge_xml_path)

    # ---------------- TURN DATA ----------------
    turn_rows = []

    for rule in TURN_RULES:
        det_key = rule["detector"]
        total_count = get_detector_total(df, det_key, "total_count")
        mean_occ = get_detector_mean(df, det_key, "mean_occupancy")

        if total_count <= 0 and mean_occ <= 0:
            continue

        from_edge = rule["from"]
        to_edges = rule["to"]
        via_map = rule.get("via", {})

        ratios = TURN_SPLIT_RATIOS.get(det_key)
        split_counts = split_count(total_count, len(to_edges), ratios)
        split_occs = split_value(mean_occ, len(to_edges), ratios)

        for to_edge, cnt, occ in zip(to_edges, split_counts, split_occs):
            turn_rows.append({
                "intersection": det_key[0],
                "detector_id": det_key[1],
                "from_edge": from_edge,
                "to_edge": to_edge,
                "via": via_map.get(to_edge, ""),
                "count": int(cnt),
                "occupancy_mean": float(occ)
            })

    turn_out_df = pd.DataFrame(turn_rows)

    if turn_out_df.empty:
        print("\nNo turn rows generated.")
        continue

    turn_agg = (
        turn_out_df.groupby(["from_edge", "to_edge", "via"], as_index=False)
        .agg(
            count=("count", "sum"),
            occupancy_mean=("occupancy_mean", "mean")
        )
        .sort_values(["from_edge", "to_edge", "via"])
    )

    print("\nTURN COUNTS + OCCUPANCY")
    print(turn_agg.to_string(index=False))

    turn_root = ET.Element("data")
    turn_interval = ET.SubElement(turn_root, "interval", {
        "id": sc["name"],
        "begin": str(sc["begin"]),
        "end": str(sc["end"])
    })

    for _, r in turn_agg.iterrows():
        attrs = {
            "from": str(r["from_edge"]),
            "to": str(r["to_edge"]),
            "count": str(int(r["count"]))
        }
        if str(r["via"]).strip():
            attrs["via"] = str(r["via"]).strip()
        ET.SubElement(turn_interval, "edgeRelation", attrs)

    turn_xml_path = OUT_DIR / f"turnData_{sc['name']}.xml"
    turn_csv_path = OUT_DIR / f"turnData_{sc['name']}.csv"
    turn_debug_path = OUT_DIR / f"turnData_debug_{sc['name']}.csv"

    turn_agg.to_csv(turn_csv_path, index=False, encoding="utf-8-sig")
    turn_out_df.to_csv(turn_debug_path, index=False, encoding="utf-8-sig")

    with open(turn_xml_path, "w", encoding="utf-8") as f:
        f.write(prettify(turn_root))

    print("Saved:", turn_csv_path)
    print("Saved:", turn_debug_path)
    print("Saved:", turn_xml_path)

print("\nDONE ✅")


SCENARIO: weekday_morning
Representative day: 2026-03-10
Saved: C:\Users\mogul\OneDrive\Masaüstü\mt_emre\detector_data_aktuell\counts_occupancy_on_edges\weekday_morning_detector_summary.csv

EDGE COUNTS + OCCUPANCY
          edge  count  occupancy_mean
   -1182936853    161       18.750000
   -1183449794     52       44.000000
-13831317#2.34    102       41.875000
   -14811373#0      8        1.250000
   -14811383#2    177       24.125000
  -202157700#5    463       34.250000
    -202157703    383       28.000000
    -202234344    732       62.250000
    -237921040   1098       22.250000
   -25563346#2     35        9.500000
-26202222#0.41    336       47.125000
   -30329931#2    579       61.750000
   202157700#3    410       15.375000
   202234068#0    520       31.625000
  237921036.41    474       32.333333
    26202222#0    260       14.000000
      75642281    363       19.750000
Saved: C:\Users\mogul\OneDrive\Masaüstü\mt_emre\detector_data_aktuell\counts_occupancy_on_edges\edge

**RandomTrips** 

& "C:\Users\mogul\AppData\Local\Programs\Python\Python39\python.exe" `
  "C:\Program Files (x86)\Eclipse\Sumo\tools\randomTrips.py" `
  -n "C:\Users\mogul\OneDrive\Masaüstü\mt_emre\digital_twin\simulation\landau_corridor.net.xml" `
  -b 0 `
  -e 3600 `
  -p 0.02 `
  --seed 42 `
  --weights-prefix "C:\Users\mogul\OneDrive\Masaüstü\mt_emre\detector_data_aktuell\new_routeSampler\allowed_weights" `
  --fringe-factor max `
  --random-routing-factor 1.5 `
  --min-distance 50 `
  --validate `
  --remove-loops `
  -o "C:\Users\mogul\OneDrive\Masaüstü\mt_emre\detector_data_aktuell\new_routeSampler\randomtrips\candidate_trips.xml" `
  --route-file "C:\Users\mogul\OneDrive\Masaüstü\mt_emre\detector_data_aktuell\new_routeSampler\randomtrips\candidate_routes.rou.xml"

**RouteSampler** **WEEKDAYMORNING**
& "C:\Users\mogul\AppData\Local\Programs\Python\Python39\python.exe" `
"C:\Program Files (x86)\Eclipse\Sumo\tools\routeSampler.py" `
-r "C:\Users\mogul\OneDrive\Masaüstü\mt_emre\detector_data_aktuell\new_routeSampler\merged\candidate_routes_merged_dedup.rou.xml" `
--edgedata-files "C:\Users\mogul\OneDrive\Masaüstü\mt_emre\detector_data_aktuell\counts_occupancy_on_edges\edgeData_weekday_morning.xml" `
--turn-files "C:\Users\mogul\OneDrive\Masaüstü\mt_emre\detector_data_aktuell\counts_occupancy_on_edges\turnData_weekday_morning.xml" `
--optimize full `
--min-count 2 `
--minimize-vehicles 1 `
--attributes 'departLane="best" departSpeed="max" type="trafficMix"' `
-o "C:\Users\mogul\OneDrive\Masaüstü\mt_emre\detector_data_aktuell\new_routeSampler\sampled\weekday_morning_sampled.rou.xml"


****WEEKDAY EVENING**
& "C:\Users\mogul\AppData\Local\Programs\Python\Python39\python.exe" `
"C:\Program Files (x86)\Eclipse\Sumo\tools\routeSampler.py" `
-r "C:\Users\mogul\OneDrive\Masaüstü\mt_emre\detector_data_aktuell\new_routeSampler\merged\candidate_routes_merged_dedup.rou.xml" `
--edgedata-files "C:\Users\mogul\OneDrive\Masaüstü\mt_emre\detector_data_aktuell\counts_occupancy_on_edges\edgeData_weekday_evening.xml" `
--turn-files "C:\Users\mogul\OneDrive\Masaüstü\mt_emre\detector_data_aktuell\counts_occupancy_on_edges\turnData_weekday_evening.xml" `
--optimize full `
--min-count 2 `
--minimize-vehicles 1 `
--attributes 'departLane="best" departSpeed="max" type="trafficMix"' `
-o "C:\Users\mogul\OneDrive\Masaüstü\mt_emre\detector_data_aktuell\new_routeSampler\sampled\weekday_evening_sampled.rou.xml"

**WEEKEND MORNING** 

& "C:\Users\mogul\AppData\Local\Programs\Python\Python39\python.exe" `
"C:\Program Files (x86)\Eclipse\Sumo\tools\routeSampler.py" `
-r "C:\Users\mogul\OneDrive\Masaüstü\mt_emre\detector_data_aktuell\new_routeSampler\merged\candidate_routes_merged_dedup.rou.xml" `
--edgedata-files "C:\Users\mogul\OneDrive\Masaüstü\mt_emre\detector_data_aktuell\counts_occupancy_on_edges\edgeData_weekend_morning.xml" `
--turn-files "C:\Users\mogul\OneDrive\Masaüstü\mt_emre\detector_data_aktuell\counts_occupancy_on_edges\turnData_weekend_morning.xml" `
--optimize full `
--min-count 2 `
--minimize-vehicles 1 `
--attributes 'departLane="best" departSpeed="max" type="trafficMix"' `
-o "C:\Users\mogul\OneDrive\Masaüstü\mt_emre\detector_data_aktuell\new_routeSampler\sampled\weekend_morning_sampled.rou.xml"

**WEEKEND EVENING**

& "C:\Users\mogul\AppData\Local\Programs\Python\Python39\python.exe" `
"C:\Program Files (x86)\Eclipse\Sumo\tools\routeSampler.py" `
-r "C:\Users\mogul\OneDrive\Masaüstü\mt_emre\detector_data_aktuell\new_routeSampler\merged\candidate_routes_merged_dedup.rou.xml" `
--edgedata-files "C:\Users\mogul\OneDrive\Masaüstü\mt_emre\detector_data_aktuell\counts_occupancy_on_edges\edgeData_weekend_evening.xml" `
--turn-files "C:\Users\mogul\OneDrive\Masaüstü\mt_emre\detector_data_aktuell\counts_occupancy_on_edges\turnData_weekend_evening.xml" `
--optimize full `
--min-count 2 `
--minimize-vehicles 1 `
--attributes 'departLane="best" departSpeed="max" type="trafficMix"' `
-o "C:\Users\mogul\OneDrive\Masaüstü\mt_emre\detector_data_aktuell\new_routeSampler\sampled\weekend_evening_sampled.rou.xml"